# Практическая работа №1. Введение в инструментарий анализа данных

## 0. Назначение практической работы

Цель работы - освоить полный базовый цикл анализа табличных данных: подготовить воспроизводимое Python-окружение, загрузить данные, проверить их структуру и качество, выполнить предобработку, получить описательные статистики, сформировать простые аналитические срезы, визуализировать закономерности и сохранить подготовленный набор данных в обменных и аналитических форматах.

В отличие от работы, построенной вокруг отдельных конструкций языка Python, здесь синтаксис используется только как инструмент решения аналитической задачи. Каждая операция должна сопровождаться интерпретацией результата.

Студент отвечает не только на вопрос «что вернула функция?», но и на вопрос «что это означает для исследуемого набора данных?».

In [ ]:
print("Hello, World!")


## 1. Постановка задачи

### 1.1. Источник датасета

**Название файла:** `The number of crimes in Russia 2008-2023.csv`

**Источник:** [СДО РТУ МИРЭА](https://online-edu.mirea.ru/)

**Дата получения:** 08.09.2026

**Период данных:** 2008-2023 (16 лет)

**Охват:** Российская Федерация в целом

### 1.2. Цель анализа

**Главная цель:** Провести анализ данных о количестве зарегистрированных преступлений в России за период 2008-2023 годов для выявления ключевых тенденций и закономерностей.

**Конкретные задачи:**

1. **Анализ временных трендов** - проследить, как менялось общее число преступлений по годам (2008-2023).

2. **Сравнение типов преступлений** - выявить наиболее и наименее распространённые категории преступлений.

3. **Региональный анализ** - определить регионы с наибольшей и наименьшей преступностью в абсолютных числах и на 100 000 населения.

4. **Анализ структуры** - рассчитать долю каждого типа преступления в общем объёме.

5. **Проверка качества данных** - выявить пропуски, аномалии, особенности.

### 1.3. Краткая предметная постановка

**Предметная область:** Криминология и статистика правонарушений.

**Объект исследования:** Зарегистрированные преступления на территории Российской Федерации.

**Ключевые вопросы исследования:**

1. Снижается ли преступность в России в целом за 16 лет?
2. Какие виды преступлений доминируют?
3. Наблюдается ли региональное неравенство?
4. Как отличается динамика разных типов преступлений?

**Ограничения данных:**
- Отсутствуют данные по Крыму и Севастополю до 2014 г.
- Нет данных по экономическим преступлениям за 2008–2009 гг.
- Отсутствует информация о раскрываемости преступлений.
- Нет данных о демографической структуре регионов.

## 2. Паспорт данных

В данном разделе формируется паспорт набора данных - формальное описание его структуры, состава и характеристик. Паспорт данных позволяет зафиксировать исходное состояние набора, оценить его пригодность для анализа и определить направления дальнейшей предобработки.

In [ ]:
from pathlib import Path
import pandas as pd


DATA_PATH = Path("The number of crimes in Russia 2008-2023.csv")
df = pd.read_csv(DATA_PATH,
                 delimiter=",",
                 decimal=".",
                 na_values="-200")

df.describe(include="all").round(2).T


### 2.1. Размер

In [ ]:
print(df.shape)
print(df.columns.tolist())
print(f"{DATA_PATH.stat().st_size / 1024:.2f} KiB")
df.head()


| Параметр | Значение |
|----------|----------|
| Число строк | 12,384 |
| Число столбцов | 5 |
| Размер файла | 589.7 KB |

### 2.2. Признаки

**Всего признаков:** 5

- **Категориальные (2):** Type of crime, Region
- **Числовые (3):** Year, The number of crimes, The number of crimes (cases per 100.000 population)

### 2.3. dtypes - типы данных

In [ ]:
import io


buf = io.StringIO()
df.info(buf=buf, memory_usage="deep")

print(buf.getvalue())


| Признак | Тип | Количество |
|---------|-----|----------|
| Type of crime | object | 12384 |
| Region | object | 12384 |
| Year | int64 | 12384 |
| The number of crimes | float64 | 12108 |
| The number of crimes (cases per 100.000 population) | float64 | 12092 |

Объём памяти: 2 MB

### 2.4. Словарь полей

| Поле | Тип | Единица | Описание | Пример |
|------|-----|---------|----------|--------|
| Type of crime | object | - | Тип преступления (8 категорий) | Murder, Theft |
| Region | object | - | Субъект РФ (включая "Russian Federation" как агрегат) | Moscow, Tuva Republic |
| Year | int | год | Год наблюдения (2008-2023) | 2008, 2015, 2023 |
| The number of crimes | float | штук | Абсолютное число зарегистрированных преступлений | 19740, 1326342 |
| The number of crimes (cases per 100.000 population) | float | на 100 тыс. | Число преступлений на 100 000 населения | 13.8, 929.2 |

### 2.5. Выводы

Набор данных представляет собой структурированную панель вида (тип преступления × регион × год), содержащую 12 384 строк и 5 признаков за период 2008-2023 гг. Данные **пригодны для разведочного анализа**, а структура панели позволяет отвечать на ключевые исследовательские вопросы о временных трендах, региональных различиях и сравнении типов преступлений; детальная проверка качества и оптимизация будут выполнены в следующем разделе.

## 3. Качество и предобработка

В данном разделе проводится детальная проверка качества данных: анализ пропусков, дубликатов, корректности типов данных, а также обоснованное преобразование признаков для дальнейшего анализа.

### 3.1. Пропуски, дубликаты и уникальные значения

In [ ]:
missing = pd.DataFrame({
    "Пропусков": df.isna().sum(),
    "Доля (%)": (df.isna().sum() / len(df) * 100).round(2)
})
print(missing)

print(f"\nВсего строк: {len(df):,}")
print(f"Строк с пропусками: {df.isna().any(axis=1).sum():,}")

print("\nПолных дубликатов:", df.duplicated().sum())

print("\nЧисло уникальных значений:")
print(df.nunique())


В наборе данных обнаружено _292 пропусков_ (2.36% строк), все - в двух числовых столбцах: `The number of crimes` и `The number of crimes (cases per 100.000 population)`. В категориальных и временном столбцах пропусков нет.

Пропуски решено НЕ заполнять и НЕ удалять, поскольку:

1. **Заполнение исказит данные** - например, если поставить `0`, то это будет означать «преступлений не было», что неверно; на самом деле данных просто нет.
2. **Удаление строк приведёт к потере информации** о других типах преступлений и годах.
3. **Пропуски имеют объективную историческую природу** - они отражают реальную ситуацию с данными, а не ошибку сбора.

Пропуски будут автоматически исключены при агрегации (например, `groupby().mean()` игнорирует NaN) и при расчёте корреляций.

Проверка показала _полное отсутствие дубликатов_, каждая комбинация уникальна и встречается в наборе ровно один раз. Такая структура соответствует «панельному» формату данных и не требует удаления каких-либо строк.

### 3.2. Типы данных, преобразования и их обоснование

| Столбец | Текущий dtype | Проблема |
|---------|---------------|----------|
| Type of crime | object | Избыточно по памяти |
| Region | object | Избыточно по памяти |
| Year | int64 | Можно на int16 |
| The number of crimes | float64 | оставлен как `float64`, поскольку содержит NaN |
| The number of crimes (cases per 100.000) | float64 | оставлен как `float64`, поскольку содержит дробные значения |

1. **`Type of crime` → `category`**:
   - Всего 9 уникальных значений, повторяются 12 384 раза
   - В `object` каждая строка хранится как Python-строка
   - В `category` хранятся только коды (int8) + таблица категорий

2. **`Region` → `category`**:
   - 86 уникальных значений, повторяются многократно
   - Аналогично `Type of crime`, экономия существенная

3. **`Year` → `int16`**:
   - Диапазон 2008-2023 помещается в int16 (от -32 768 до 32 767)
   - `int64` избыточен (8 байт вместо 2)

In [ ]:
before = df.memory_usage(deep=True).sum()

clean_df = df.copy()

# Преобразование категориальных признаков
clean_df["Type of crime"] = clean_df["Type of crime"].astype("category")
clean_df["Region"] = clean_df["Region"].astype("category")

# Оптимизация Year
clean_df["Year"] = clean_df["Year"].astype("int16")

after = clean_df.memory_usage(deep=True).sum()

print(f"До: {before / 1024**2:.2f} MiB")
print(f"После: {after / 1024**2:.2f} MiB")
print(f"Снижение: {(1 - after / before) * 100:.1f}%")


### 3.3. Создание вычисляемого признака и двух отдельных DataFrame

In [ ]:
# Проверить, велико ли количество преступлений.
clean_df["is_high_crime"] = clean_df["The number of crimes"] > 1000

clean_df.head()


Столбец `Region` содержит **два уровня агрегации**:
- **Национальный уровень:** `Russian Federation` (144 строки = 9 типов × 16 лет)
- **Региональный уровень:** 85 субъектов РФ (12 240 строк)

**Решение:** создать **два отдельных DataFrame**:

1. **`clean_df`** - полный набор (12 384 строки), используется для 
   анализа общенациональных трендов и общего объёма.
   
2. **`df_regions`** - только регионы (12 240 строк), используется 
   для регионального анализа, где **недопустимо** смешивать 
   агрегат с отдельными субъектами.

3. **`df_country`** - только `Russian Federation` (144 строки), 
   используется для анализа **национальных трендов** по типам 
   преступлений.

Такое разделение позволяет **избежать двойного счёта** и 
корректно интерпретировать результаты на разных уровнях 
агрегации.

In [ ]:
# df_regions - только для субъектов
df_regions = clean_df[clean_df["Region"] != "Russian Federation"].copy()

print(f"clean_df:    {len(clean_df):,} строк")
print(f"df_regions:  {len(df_regions):,} строк")
print(f"Разница:     {len(clean_df) - len(df_regions):,} строк (= 9 типов × 16 лет)")


In [ ]:
# df_country - для всей России
df_country = clean_df[clean_df["Region"] == "Russian Federation"].copy()

print(f"df_country: {len(df_country):,} строк")
print(f"\nТипы преступлений:")
print(df_country["Type of crime"].value_counts())


### 3.4. Выводы

Выполнена минимальная, но обоснованная предобработка: проверено 276 пропусков, проверены дубликаты, оптимизированы типы данных и получилась экономия 87.9% памяти, данные сохранены в новой переменной `clean_df`, исходный `df` не изменён. Созданы два отдельных DataFrame для субъектов и для общей России.

## 4. EDA - Разведочный анализ

Разведочный анализ данных (EDA) направлен на выявление закономерностей, трендов и аномалий в наборе данных о преступлениях в России за 2008-2023 гг. В данном разделе выполняются: анализ распределения по типам преступлений, групповые агрегации, условная фильтрация, корреляционный анализ и визуализации.

### 4.1. Распределение наблюдений по типам преступлений

Первым шагом анализа является оценка **сбалансированности** набора данных по категориальному признаку `Type of crime`. Это важно для понимания структуры выборки и потенциальных проблем при моделировании.

In [ ]:
type_stats = (
    df_regions["Type of crime"]
        .value_counts()
        .to_frame("n")
        .assign(share_pct=lambda x: x["n"] / len(df_regions) * 100)
        .round(2)
)

print(type_stats)


Набор данных **идеально сбалансирован** по типам преступлений: каждый из 9 типов представлен ровно 12 240 наблюдениями (11.11%). Это объясняется структурой панели: (9 типов × 85 регионов × 16 лет) = 12 240.

### 4.2. Групповые агрегации: типы преступлений

Следующий шаг - сравнить **абсолютное число преступлений** по типам. Это покажет, какие категории доминируют по объёму, а какие представлены минимально.

In [ ]:
# Группировка по типу преступления
crime_by_type = (
    df_regions.groupby("Type of crime", observed=True)
        .agg(
            total_crimes=("The number of crimes", "sum"),
            mean_crimes=("The number of crimes", "mean"),
            median_crimes=("The number of crimes", "median"),
            max_crimes=("The number of crimes", "max"),
            n_observations=("The number of crimes", "count")
        )
        .sort_values("total_crimes", ascending=False)
        .round(2)
)

print(crime_by_type)


В `df_regions` наблюдается **экстремальный дисбаланс**: Theft (14.2 млн) превышает Rape (65 тыс.) в **219 раз**. Средние значения в 1.5-3 раза больше медиан, что указывает на **асимметричные распределения** - вклад крупных регионов (Москва, Московская область) непропорционален. Число наблюдений: 1348 для большинства типов, но 1182 для экономических преступлений (пропуски за 2008-2009 гг.). Для корректного сравнения типов между собой требуется **нормализация** (к % от 2008 г. или к доле в общем объёме).

### 4.3. Условная фильтрация

Для иллюстрации составного фильтра через `loc` выберем наблюдения, удовлетворяющие **двум условиям одновременно**:
- тип преступления = Murder
- число преступлений > 100

Это позволит выделить регионы с **высоким абсолютным уровнем** убийств.

In [ ]:
# Составной фильтр: Murder & число > 100
subset = df_regions.loc[
    (df_regions["Type of crime"] == "Murder") & 
    (df_regions["The number of crimes"] > 100),
    ["Type of crime", "Region", "Year", "The number of crimes"]
]

print(f"Строк в срезе: {len(subset)}")
print("\nПримеры:")
print(subset.head(10).to_string(index=False))


Фильтр `Murder & > 100` выделил **625 наблюдений** (4.6% всех Murder-записей). В выборку попали **крупные регионы** (Moscow Oblast - 1011, Sverdlovsk, Krasnodar), малые регионы (Чукотка, Ненецкий АО) - нет. Преобладают данные за **2008-2010 гг.**, когда уровень убийств был максимальным.

### 4.4. Корреляция

In [ ]:
corr = df_regions[["Year", "The number of crimes", 
                  "The number of crimes (cases per 100.000 population)"]].corr()

display(corr.round(3))


Корреляционный анализ выявил **три закономерности**:

1. **Year ↔ Crimes (r = -0.078):** Слабая отрицательная связь - временной тренд **размывается** региональной неоднородностью.

2. **Year ↔ Rate (r = -0.104):** Слабая отрицательная связь - аналогично, тренд не виден на уровне отдельных наблюдений.

3. **Crimes ↔ Rate (r = 0.683):** **Умеренная положительная** связь - ожидаемо, поскольку rate рассчитывается из crimes, но не равно 1.0 из-за разного размера регионов.

Сильных линейных связей между признаками нет. Временной тренд преступности **не проявляется** в попарной корреляции - его следует анализировать **отдельно** через группировку по годам.

### 4.5. Правило IQR

In [ ]:
q1 = df_regions["The number of crimes"].quantile(0.25)
q3 = df_regions["The number of crimes"].quantile(0.75)

iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

mask = (
    (df_regions["The number of crimes"] < lower) |
    (df_regions["The number of crimes"] > upper)
)

mask_sum = mask.sum()
percentage = mask_sum / len(df_regions) * 100

print(f"Q1={q1:.3f}; Q3={q3:.3f}; IQR={iqr:.3f}")
print(f"Границы: [{lower:.3f}; {upper:.3f}]")
print(f"Кандидатов: {mask_sum} ({percentage:.2f}% строк)")


Правило 1,5·IQR пометило **1 845 наблюдений (15.07%)** как кандидатов на выброс (Q1=37, Q3=618, IQR=581, верхняя граница = 1489.5). Доля **слишком велика** для автоматического удаления - нормальные выбросы обычно составляют 1-5%.

Причина - **правая асимметрия** распределения: mean (10 537) > median (6 756) в 1.5 раза. Кандидаты - это **крупные регионы** (Москва, Московская область, Краснодарский край), которые **объективно** имеют больше преступлений из-за размера населения.

Удалять эти строки **нельзя** - это исказит данные. Правило IQR создаёт **список для проверки**, но **не принимает решение** об ошибочности наблюдений.

### 4.6. Визуализации

#### Bar chart - Топ-10 регионов по числу преступлений:

In [ ]:
import matplotlib.pyplot as plt


# Top 10 регионов по числу преступлений
top_regions = (
    df_regions.groupby('Region', observed=True)['The number of crimes']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(10, 6))

ax = top_regions.plot(kind='barh', color='steelblue', edgecolor='black')
ax.set_title('Топ-10 регионов по числу преступлений (2008–2023)', 
             fontsize=13)
ax.set_xlabel('Суммарное число преступлений', fontsize=11)
ax.set_ylabel('Регион', fontsize=11)
plt.grid(True, axis="x", linestyle="-", alpha=0.7)
ax.invert_yaxis()  # Чтобы наибольшее значение было сверху

plt.tight_layout()
plt.show()


Диаграмма показывает **резкую региональную неравномерность**: Топ-10 регионов концентрируют значительную долю всех преступлений. **Москва** и **Московская область** доминируют с большим отрывом - это объясняется **численностью населения** (более 12 млн и 8 млн человек соответственно) и высокой плотностью.

Однако **абсолютные значения не отражают уровень криминогенности**: крупный регион естественно имеет больше преступлений. Для корректного сравнения необходим переход к **относительным показателям** на 100000 населения.

#### Histogram - Распределение `The number of crimes`:

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df_regions['The number of crimes'].dropna(), 
         bins=50, edgecolor='white', color='steelblue')

plt.title('Распределение числа преступлений по наблюдениям', fontsize=13)
plt.xlabel('Число преступлений', fontsize=11)
plt.ylabel('Частота', fontsize=11)
plt.yscale('log')  # Логарифмическая шкала для наглядности

plt.tight_layout()
plt.show()


Распределение имеет **выраженную правую асимметрию** (right-skewed): подавляющее большинство наблюдений сосредоточено в диапазоне **0-500 преступлений**, но есть длинный хвост до **1 000 000+** (Московская область, 2008 г.).

**Логарифмическая шкала** по оси Y позволяет увидеть структуру распределения: даже редкие значения (> 10000) встречаются **регулярно**, что подтверждает - это не аномалии, а **естественная вариация** между крупными и малыми регионами.

Распределение **не является нормальным** (mean = 10 537 при median = 6 756). Для статистического анализа требуются **непараметрические методы** или **логарифмирование**.

#### Scatter plot - Absolute vs Rate per 100k:

In [ ]:
plt.figure(figsize=(10, 6))

# Подвыборка для наглядности
sample = df_regions.dropna(subset=['The number of crimes', 
                                    'The number of crimes (cases per 100.000 population)']).sample(
    n=3000, random_state=42
)

plt.scatter(
    sample['The number of crimes'],
    sample['The number of crimes (cases per 100.000 population)'],
    s=10, alpha=0.4, color='steelblue'
)

plt.title('Связь абсолютного числа и интенсивности преступлений', fontsize=13)
plt.xlabel('Число преступлений (абсолютное)', fontsize=11)
plt.ylabel('Число преступлений на 100 000 населения', fontsize=11)
plt.xscale('log')  # Логарифмическая шкала

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Точечная диаграмма показывает **умеренную положительную связь** (r ≈ 0.683): чем больше абсолютное число преступлений, тем выше интенсивность на 100 000 населения. Однако точки образуют **широкое облако**, а не узкую линию - это указывает на **влияние размера региона**.

**Ключевое наблюдение:** при одинаковом абсолютном числе преступлений (например, ~1000) значения rate варьируются **от 20 до 500+** - в 25 раз. Это означает, что **абсолютное число нельзя использовать** для сравнения криминогенности регионов без учёта населения.

Для сравнения регионов необходим **rate per 100 000** - он нормирует данные и устраняет эффект размера.

#### Time series - Динамика по годам:

In [ ]:
# Динамика по годам для каждого типа преступления
crime_trend = (
    df_regions.groupby(['Year', 'Type of crime'], observed=True)
    ['The number of crimes']
    .sum()
    .unstack()
)

# Нормализация: 2008 = 100%
crime_norm = crime_trend.div(crime_trend.iloc[0]) * 100

plt.figure(figsize=(14, 7))
for crime_type in crime_norm.columns:
    plt.plot(crime_norm.index, crime_norm[crime_type], 
             marker='o', label=crime_type, linewidth=1.5, markersize=4)

plt.axhline(y=100, color='black', linestyle='--', alpha=0.5, 
            label='Уровень 2008 г.')
plt.title('Относительная динамика преступлений по типам (2008 = 100%)', 
          fontsize=13)
plt.xlabel('Год', fontsize=11)
plt.ylabel('Индекс (% от уровня 2008 г.)', fontsize=11)

plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Нормализованный график (2008 = 100%) позволяет сравнить **темпы изменений** независимо от абсолютного масштаба. Выявлены **три принципиально разные траектории**:

**1. Резкое снижение (>50%):**
- **Robbery without violence:** -91% (с 100% до 9%)
- **Hooliganism:** -88%
- **Murder:** -62%
- **Theft:** -56%

**2. Умеренное снижение (30-50%):**
- **Rape:** -50%
- **Robbery with violence:** -50%
- **Intentional Bodily Harm:** -47%

**3. Относительная стабильность:**
- **Extortion:** -10% (практически вернулось к уровню 2008 г.)
- **Crimes in Economic Activity:** стабильно ~100% (нет данных за 2008-2009, базовая точка - 2010)

**Все типы преступлений** показали снижение, но **темпы радикально различаются** - от 10% до 91%. Наибольший спад пришёлся на **2008-2015 гг.**, после чего динамика замедлилась. Наиболее устойчивыми оказались **экономические преступления** и **вымогательство**.

### 4.7. Выводы

Разведочный анализ подтвердил, что набор данных **пригоден для анализа** и обладает высокой аналитической ценностью. Выявлены **три ключевые закономерности**:

- Экстремальный дисбаланс по объёму - Theft (14.2 млн) превышает Rape (65 тыс.) в 219 раз;
- выраженная правая асимметрия распределений - mean > median в 1.5-3 раза, что требует нормализации при визуализации;
- принципиально разная динамика по типам - от снижения на 91% (Robbery without violence) до относительной стабильности (Extortion).

Корреляционный анализ показал слабую связь между годом и числом преступлений (r ≈ −0.08), что объясняется региональной неоднородностью; правило IQR пометило 15% наблюдений как кандидатов на выбросы, однако они отражают **естественную вариацию** крупных регионов, а не ошибки. Структура панели позволяет отвечать на ключевые исследовательские вопросы о временных трендах, региональных различиях и сравнении типов преступлений.

## 5. Экспорт

### 5.1. CSV и Parquet

После завершения предобработки и разведочного анализа необходимо сохранить подготовленный набор данных во внешние файлы. Экспорт преследует **три цели**: обеспечить воспроизводимость анализа, создать обменные файлы для дальнейшей работы и сравнить форматы хранения по размеру и сохранению схемы. Для корректного сравнения один и тот же DataFrame `clean_df` будет записан в **трёх форматах**: CSV, Parquet и JSON.

In [ ]:
from pathlib import Path


OUT = Path("output")
OUT.mkdir(exist_ok=True)

clean_df.to_csv(
    OUT / "CrimesRu_clean.csv",
    index=False,
    encoding="utf-8"
)

clean_df.to_json(
    OUT / "CrimesRu_clean.json",
    orient="records",
    lines=True,
    force_ascii=False
)

clean_df.to_parquet(
    OUT / "CrimesRu_clean.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy"
)


### 5.2. Размеры

После записи необходимо измерить **фактические размеры** файлов на диске. Это позволит количественно оценить эффективность каждого формата. Сравнение проводится на **одних и тех же данных**, различается только формат хранения.

In [ ]:
files = [
    OUT / "CrimesRu_clean.csv",
    OUT / "CrimesRu_clean.json",
    OUT / "CrimesRu_clean.parquet"
]

for path in files:
    size_mib = path.stat().st_size / 1024**2
    print(f"{path.name:24s} {size_mib:7.2f} MiB")


**Parquet** оказался наиболее компактным благодаря колоночному хранению и сжатию `snappy`. **JSON** занимает больше всего места из-за избыточности ключей в каждой строке. **CSV** - промежуточный вариант, удобный для обмена.

### 5.3. Проверка повторного чтения

Размер - **не единственный критерий**. Не менее важно, как формат сохраняет **типы данных**. Это будет проверено через процедуру round-trip.

**Round-trip** (цикл «записать → прочитать → проверить») позволяет убедиться, что файлы действительно пригодны для продолжения аналитического pipeline. Особое внимание уделяется **сохранению dtypes**: CSV - текстовый формат, при чтении типы восстанавливаются заново; Parquet - бинарный, хранит схему вместе с данными.

In [ ]:
csv_back = pd.read_csv(OUT / "CrimesRu_clean.csv")
json_back = pd.read_json(OUT / "CrimesRu_clean.json", lines=True)
parquet_back = pd.read_parquet(OUT / "CrimesRu_clean.parquet")

print("=" * 60)
print("СРАВНЕНИЕ SHAPE")
print("=" * 60)
print(f"Оригинал:  {clean_df.shape}")
print(f"CSV:       {csv_back.shape}")
print(f"Parquet:   {parquet_back.shape}")
print(f"JSON:      {json_back.shape}")

print("\n" + "=" * 60)
print("СРАВНЕНИЕ DTYPES")
print("=" * 60)
print("CSV после чтения:")
print(csv_back.dtypes)
print("\nParquet после чтения:")
print(parquet_back.dtypes)


Все три формата **успешно прошли round-trip**: число строк и столбцов совпадает с оригиналом (12 384 × 6). Однако **dtypes различаются**.

**Parquet сохраняет схему полностью**, тогда как **CSV теряет типы** - категориальные признаки превращаются в `object`, а `int16` - в `int64`. Это означает, что после чтения CSV потребуется **повторная оптимизация dtypes**.

Для аналитического pipeline **Parquet предпочтительнее**: он сохраняет типы, занимает в 3.6 раз меньше места и поддерживает выборочное чтение столбцов. **CSV** остаётся удобным для обмена и просмотра в текстовом редакторе. **JSON** - оптимален для полуструктурированного взаимодействия, но избыточен по размеру.

### 5.4. Выводы

Экспорт подготовленного набора данных выполнен в трёх форматах - CSV, Parquet и JSON, - каждый из которых имеет свои преимущества и ограничения.

**Parquet** оказался наиболее эффективным для аналитического pipeline: он занимает в 3.6 раз меньше места, чем CSV (0.42 MiB против 1.53 MiB), полностью сохраняет схему данных (типы `category` и `int16` остаются неизменными после round-trip) и поддерживает выборочное чтение отдельных столбцов, что критично при работе с большими объёмами.

**CSV** остаётся удобным для обмена и просмотра в текстовом редакторе, однако теряет типы при чтении - категориальные признаки восстанавливаются как `object`, что требует повторной оптимизации dtypes.

**JSON** обеспечивает гибкость полуструктурированного формата, но занимает в 2.5 раза больше места, чем CSV, из-за избыточности ключей.

Проверка round-trip подтвердила, что все три файла корректно восстанавливают исходные данные (12 384 строки × 5 столбцов), что делает их пригодными для дальнейшего использования. Для последующих задач курса, включая распределённую обработку в Spark, рекомендуется использовать **Parquet** как основной аналитический формат.

## 6. Итог

### 6.1. Основные выводы

**1. Набор данных представляет собой сбалансированную панель** вида (тип преступления × регион × год) объёмом 12 384 строки и 5 признаков за 2008–2023 гг. Каждый из 9 типов преступлений представлен одинаковым числом наблюдений (1 360), что делает набор удобным для сравнительного анализа.

**2. Данные содержат объективные пропуски** (276 в столбце `The number of crimes` и 292 в столбце rate), связанные с историческими причинами: отсутствием статистики по Крыму и Севастополю до 2014 г. и по экономическим преступлениям до 2010 г. Пропуски решено не заполнять и не удалять, поскольку они отражают реальную ситуацию с данными.

**3. Оптимизация типов данных дала значительный эффект** - перевод категориальных признаков в `category` и `Year` в `int16` сократил объём памяти на **76.3%** (с 2.87 MiB до 0.68 MiB) без потери информации.

**4. Выявлен экстремальный дисбаланс по объёму преступлений** - Theft (14.2 млн) превышает Rape (65 тыс.) в **219 раз**. Распределения числовых признаков имеют **правую асимметрию** (mean > median в 1.5–3 раза), что требует нормализации при визуализации.

**5. Динамика разных типов преступлений принципиально различна** - от снижения на 91% (Robbery without violence) и 88% (Hooliganism) до относительной стабильности (Extortion - снижение на 10%). Все типы показали снижение за 16 лет, но темпы различаются радикально.

**6. Корреляционный анализ не выявил сильных линейных связей** между годом и числом преступлений (r ≈ −0.08), что объясняется высокой региональной неоднородностью. Наиболее заметная связь - между абсолютным числом и rate per 100 000 (r = 0.683), что ожидаемо, поскольку rate рассчитывается из absolute.

**7. Правило IQR пометило 15% наблюдений** как кандидатов на выбросы, однако анализ показал, что это не ошибки, а естественная вариация крупных регионов. Автоматическое удаление таких строк недопустимо.

**8. Parquet - оптимальный формат для аналитического pipeline** - он занимает в 3.6 раз меньше места, чем CSV, и полностью сохраняет схему данных при round-trip, тогда как CSV теряет типы (`category` → `object`).

### 6.2. Ограничения исследования

**1. Отсутствие временной детализации.** Данные представлены с **годовой дискретностью** - нет информации о месяцах, днях или часах. Это ограничивает анализ сезонности, краткосрочных трендов и точечных событий. Временной ряд построен только на 16 точках (2008-2023), что снижает статистическую мощность временных тестов.

**2. Смешение уровней агрегации в столбце Region.** Столбец содержит как агрегат `Russian Federation` (национальный уровень), так и 85 отдельных субъектов РФ. Это потребовало создания двух отдельных DataFrame (`df_regions` и `df_country`), однако в исходном виде столбец может вводить в заблуждение при неосторожной агрегации.

**3. Отсутствие демографических и социально-экономических показателей.** Набор не содержит данных о численности населения, уровне дохода, образовании, безработице и других факторах, которые могли бы объяснить региональные различия в преступности. Анализ ограничен **описательной статистикой** без возможности проверки причинно-следственных гипотез.

**4. Невозможность интерпретации экономических преступлений до 2010 г.** Категория `Crimes in the Sphere of Economic Activity` не имеет данных за 2008–2009 гг., что делает её **несопоставимой** с другими типами на всём временном интервале. Сравнение динамики этой категории с остальными ограничено периодом 2010–2023.

### 6.3. Заключительное замечание

Набор данных **пригоден для разведочного анализа** и позволяет отвечать на ключевые исследовательские вопросы о временных трендах, региональных различиях и структуре преступности.

Однако для полноценного причинно-следственного анализа требуется **дополнение внешними источниками** - демографической статистикой, социально-экономическими показателями и данными с более высокой временной детализацией.